# Ship Engine Anomaly Detection

Compares a statistical rule (IQR) with two unsupervised models (One-Class SVM, Isolation Forest) for flagging abnormal readings in 19,535 engine-sensor records (6 sensors).

*Origin: University of Cambridge Data Science Career Accelerator coursework (2024-25). Data source anonymised and not included. Cell outputs cleared to protect the source data; results quoted in the README come from the original run.*

### <font color="blue">**Following the Rubric Skills criteria, the project has been divided into three main parts:**</font>

* <font color="blue">**PART I:** Initial data exploration and assessment
* **PART II:**Anomaly detection: Statistical methods
* **PART III:**Anomaly detection: Machine learning (ML) models</font>

### <font color="blue">**PART I: Initial data exploration and assessment**</font>

In [ ]:
# 1. Importing required libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from IPython.core.display import display, HTML
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')
from matplotlib.colors import ListedColormap

In [ ]:
# URL to import dataset from GitHub.
url = 'data/engine_sensors.csv'

In [ ]:
# Loading the dataset
data = pd.read_csv(url)
print('Data information:')
print()
print(data.info())

 <font color="blue">**Initial data exploration and assessment**</font>

<font color="blue">**Missing and duplicate values in the data are identified**</font>

In [ ]:
# 2. Missing and duplicate values in the data are identified or reported as
# non-existent
for column in data.columns:
  print(f"Unique values in column '{column}': {data[column].nunique()}")
print()
print()
print('Checking the data shape:')
print(f"The dataset contain {data.shape[0]} rows and {data.shape[1]} columns" )
print()
print()
print('Checking data information:')
print()
print(data.info())
print()
# Checking for - Step 1. Missing and duplicate values
missing_values = data.isnull().sum()
print('Checking if there are missing values:')
print()
print(missing_values)
print()
# Checking for duplicated values
duplicated_values = data.duplicated().sum()
print('Checking if there are duplicate values:', duplicated_values)
print()

<font color="blue">**Descriptive statistics of the data set are generated**</font>

In [ ]:
# 3.1 The descriptive statistics of the data set are generated
descriptive_statistics = data.describe()
print("The dataset (data) description below:")
print()
print(descriptive_statistics)

<font color="blue">**The mean and medians for each feature are calculated**</font>

In [ ]:
# 3.2 The mean and medians for each feature are documented providing insights
mean = data.mean()
median = data.median()
print('The mean for each feature is:')
print()
print(mean)
print()
print('The median for each feature is:')
print()
print(median)
print()

<font color="blue">**Range values beyond the 95th percentile for at least two features identified**</font>

In [ ]:
# 3.3 The range values beyond the 95th percentile for at least two features are
# identified
percentile_95 = data.quantile(0.95)
print('The 95th percentile for each feature is:')
print()
print(percentile_95)
print()
print(data.columns)

In [ ]:
# Description of values beyond the 95th percentile for all features
print(data[data['Engine rpm'] > percentile_95['Engine rpm']].describe())
print()
print(data[data['Lub oil pressure'] > percentile_95['Lub oil pressure']].describe())
print()
print(data[data['Fuel pressure'] > percentile_95['Fuel pressure']].describe())
print()
print(data[data['Coolant pressure'] > percentile_95['Coolant pressure']].describe())
print()
print(data[data['lub oil temp'] > percentile_95['lub oil temp']].describe())
print()
print(data[data['Coolant temp'] > percentile_95['Coolant temp']].describe())
print()

In [ ]:
# 4.1 Visualising features' range beyond their 95th percentile
features = data.columns
print(f"Features: {features}")
print()

# Checking and counting for each row how many features have values greater
# than their respective 95th percentile
data['outliers_count_95th'] = (data > percentile_95).sum(axis=1)
print('outliers_count_95th:')
print(data['outliers_count_95th'])
print()
print('outliers_count_95th description:')
print(data['outliers_count_95th'].describe())
print()
print('outliers_count_95th information:')
print(data.info())
print()
print('outliers_count_95th value counts:')
print(data['outliers_count_95th'].value_counts())

# Visualisation of Number of Features Beyond 95th Percentile
plt.figure(figsize=(10, 6))
scatter = sns.scatterplot(x=data.index, y='outliers_count_95th', hue='outliers_count_95th',
                           data=data, palette='viridis', legend='full')

plt.title('Number of Features Beyond 95th Percentile')
plt.xlabel('Sample Index')
plt.ylabel('Number of Features Beyond 95th Percentile')

# Placing the legend outside the plot
legend = plt.legend(
    title="Features Beyond 95th Percentile",
    labels=[f'{i} Features' for i in data['outliers_count_95th'].unique()],
    bbox_to_anchor=(1.05, 1),  # Position the legend outside the plot
    loc='upper left'  # Location of the legend
)
plt.setp(legend.get_title(), fontsize=12)
[plt.setp(text, fontsize=10, fontweight='bold') for text in legend.get_texts()]

# Adjusting the layout for the legend
plt.tight_layout()
plt.show()

# samples that have at least two features above the 95th percentile
samples_with_at_least_two_outliers = data[data['outliers_count_95th'] >= 2]['outliers_count_95th']

# histogram of samples that have at least two features as outliers
plt.figure(figsize=(10, 6))
hist = sns.histplot(samples_with_at_least_two_outliers, kde=False, color='blue', edgecolor='black')

plt.title('Number of Samples with at Least Two Features Beyond 95th Percentile', fontsize=14)
plt.xlabel('Number of Features Beyond 95th Percentile', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
total_samples = len(data)
for patch in hist.patches:
    height = patch.get_height()
    percentage = (height / total_samples) * 100
    if height > 0:  # Only display labels for non-zero values
        plt.text(patch.get_x() + patch.get_width() / 2, height + 0.05, f'{int(height)} ({percentage:.2f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold', color='black')

plt.tight_layout()
plt.show()

In [ ]:
# 4.2 Corresponding histograms for each feature

n_features = len(features)
n_cols = 2  # Number of columns for subplots
n_rows = (n_features + n_cols - 1) // n_cols  # ensuring that all features are accommodated

# Figure with subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
fig.suptitle('Histograms of filtered data only including values above the 95th percentile for every feature', fontsize=14, y=1)
plt.subplots_adjust(top=1, hspace=0.5)

# Flatten axes if necessary
if n_rows > 1:
    axes = axes.flatten()

plt.subplots_adjust(top=0.85, hspace=0.5)

# Plot histograms for each feature
for i, (feature, ax) in enumerate(zip(features, axes)):
    # Filter data to only include values above the 95th percentile
    data_above_95th = data[data[feature] > percentile_95[feature]]

    # Plot histogram for values above 95th percentile
    sns.histplot(data_above_95th[feature], kde=True, color='green', edgecolor='black', ax=ax)
    ax.set_title(f'95th Percentile Histogram of {feature}', fontsize=8)
    ax.set_xlabel(feature, fontsize=8)
    ax.set_ylabel('Frequency', fontsize=8)
    # Flag the 95th percentile line
    xlim = ax.get_xlim()
    ax.set_xlim(xlim[0], xlim[1] * 1.1)

    ax.annotate(f'{percentile_95[feature]:.2f}', xy=(percentile_95[feature], 0), xytext=(percentile_95[feature] * 1.02, max(data_above_95th[feature].value_counts()) * 0.9),
            arrowprops=dict(facecolor='red', shrink=0.05),
            fontsize=10, fontweight='bold', color='red',
            ha='center', va='bottom',
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='none', facecolor='white'))

# Remove any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Adjust layout to prevent overlap and show the plot
plt.tight_layout()
plt.show()

In [ ]:
# 4.3 Visualisation of total number of features beyond the 95th percentile values
# per each feature

for column in data.columns:
    if column == 'outliers_count_95th':
        break  # Stop plotting when reaching 'outlier_count_95th'

    plt.figure(figsize=(15, 6))
    scatter = sns.scatterplot(x=data[column], y='outliers_count_95th', hue='outliers_count_95th',
                              data=data, palette='viridis', legend='full')

    # Legend and place it outside the plot
    legend = plt.legend(
        title=f"Number of Features Beyond 95th Percentile per {column} Sample",
        labels=[f'{i} Features' for i in data['outliers_count_95th'].unique()],
        bbox_to_anchor=(1.05, 1),  # Position the legend outside the plot
        loc='upper left'
    )
    plt.setp(legend.get_title(), fontsize=12)
    [plt.setp(text, fontsize=10, fontweight='bold') for text in legend.get_texts()]

    plt.tight_layout()
    plt.show()

<font color="blue">**Summary of Scatter Plots on Engine Parameters and Outliers:**</font>

**Observations:**
* Samples that show 0 to 1 feature as an outlier, suggesting normal operation.
* Higher values of engine rpm, fuel, and coolant pressure appear to be linked to samples with multiple outliers
* Lower values for lub oil temp and coolant temp appear to be linked with higher number of outliers

<font color="blue">**Visualisations: Histograms, Outliers Boxplot and Correlation Map**</font>

In [ ]:
# 4.4 Visualisations to understand of the data

# Histogram for each feature
print('Histogram for each feature')
for column in data.columns:

    if column == 'outliers_count_95th':
        break  # Stop plotting when reaching 'outlier_count_95th'

    plt.figure(figsize=(10, 5))
    sns.histplot(data[column], kde=True, color='red', edgecolor='black')
    plt.title(f'Histogram of {column}', fontsize=12)
    plt.xlabel(column, fontsize=10)
    plt.ylabel('Frequency', fontsize=10)
    plt.axvline(percentile_95[column], color='blue', linestyle='dashed', linewidth=1)
    xlim = plt.gca().get_xlim()
    plt.xlim(xlim[0], xlim[1] * 1.1)
    plt.annotate(f'{percentile_95[column]:.2f}', xy=(percentile_95[column], 0), xytext=(percentile_95[column], 10),
            arrowprops=dict(facecolor='red', shrink=0.05),
             fontsize=10, fontweight='bold', color='white',
             ha='center', va='bottom',
             bbox=dict(boxstyle="round,pad=0.3", edgecolor='none', facecolor='black'))
plt.tight_layout()
plt.show()

# Boxplots to identify outliers beyonf 95th percentl
for column in data.columns:
    if column == 'outliers_count_95th':
      break
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=data[column])
    plt.title(f'Boxplot of outliers_count_95th - in {column}', fontsize=12)
    plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Corr heatmap for outliers_count_95th
print('Correlation Map - outliers_count_95th')
# Check if 'outlier_count_95th' is in the columns before dropping
if 'outliers_count_95th' in data.columns:
    correlation_matrix = data.drop(columns=['outliers_count_95th']).corr() # Drop 'outlier_count_95th'
else:
    correlation_matrix = data.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap of Engine Features - outliers_count_95th', fontsize=12)
plt.show()

# Corr heatmap with all data
print('Correlation Map')
if 'outliers_count_95th' in data.columns:
    correlation_matrix = data.drop(columns=['outliers_count_95th']).corr() # Drop 'outlier_count_95th'
else:
  correlation_matrix = data.corr() # full data
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', annot_kws={"size": 10}, linewidths=0.5, linecolor='gray')
plt.title('Correlation Heatmap of Engine Features - All Data Points', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# 4.5 Further visualisations to understand the data

# Histogram for each feature
print('Histogram for each feature')
for column in data.columns:
    if column == 'outliers_count_95th':
        break

    plt.figure(figsize=(10, 5))
    sns.histplot(data[column], kde=True, color='red', edgecolor='black')
    plt.title(f'Histogram of {column}', fontsize=12)
    plt.xlabel(column, fontsize=10)
    plt.ylabel('Frequency', fontsize=10)

    # Calculate mean, Q1 and Q3
    mean_val = data[column].mean()
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)

    # Plot dashed lines for percentile 95, mean, Q1 and Q3
    plt.axvline(percentile_95[column], color='blue', linestyle='dashed', linewidth=1, label='95th Percentile')
    plt.axvline(mean_val, color='green', linestyle='dashed', linewidth=1, label='Mean')
    plt.axvline(Q1, color='orange', linestyle='dashed', linewidth=1, label='Q1')
    plt.axvline(Q3, color='purple', linestyle='dashed', linewidth=1, label='Q3')

    xlim = plt.gca().get_xlim()
    plt.xlim(xlim[0], xlim[1] * 1.1) # room for annotations

    # Annotations
    plt.annotate(f'{percentile_95[column]:.2f}', xy=(percentile_95[column], 0), xytext=(percentile_95[column], 0.6 * plt.ylim()[1]),
                 arrowprops=dict(facecolor='red', shrink=0.05),
                 fontsize=10, fontweight='bold', color='white',
                 ha='center', va='bottom',
                 bbox=dict(boxstyle="round,pad=0.3", edgecolor='none', facecolor='black'))

    plt.annotate(f'{mean_val:.2f}', xy=(mean_val, 0), xytext=(mean_val, 0.7 * plt.ylim()[1]),
                 arrowprops=dict(facecolor='green', shrink=0.05),
                 fontsize=10, fontweight='bold', color='white',
                 ha='center', va='bottom',
                 bbox=dict(boxstyle="round,pad=0.3", edgecolor='none', facecolor='black'))

    plt.annotate(f'{Q1:.2f}', xy=(Q1, 0), xytext=(Q1, 0.8 * plt.ylim()[1]),
                 arrowprops=dict(facecolor='orange', shrink=0.05),
                 fontsize=10, fontweight='bold', color='white',
                 ha='center', va='bottom',
                 bbox=dict(boxstyle="round,pad=0.3", edgecolor='none', facecolor='black'))

    plt.annotate(f'{Q3:.2f}', xy=(Q3, 0), xytext=(Q3, 0.9 * plt.ylim()[1]),
                 arrowprops=dict(facecolor='purple', shrink=0.05),
                 fontsize=10, fontweight='bold', color='white',
                 ha='center', va='bottom',
                 bbox=dict(boxstyle="round,pad=0.3", edgecolor='none', facecolor='black'))

    plt.legend()
    plt.tight_layout()
    plt.show()

### <font color="blue">**PART II**: **Anomaly detection: Statistical methods**</font>

In [ ]:
# 5. The Interquartile Range (IQR) method was used to detect outliers in the data

# Function to calculate IQR outliers
def iqr_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    is_outlier = (series < lower_bound) | (series > upper_bound)
    return is_outlier, Q1, Q3

# Apply the IQR outlier detection method to each feature
outlier_counts = {}
for column in data.columns:
    if 'outlier' not in column:  # Avoid processing the outlier columns again
        data[column + '_is_outlier'], Q1, Q3 = iqr_outliers(data[column])
        outlier_counts[column] = data[column + '_is_outlier'].sum()
        print(f'For {column}:')
        print(f'Q1: {Q1}')
        print(f'Q3: {Q3}')
        print(f'Number of outliers: {outlier_counts[column]}')
        print()

# Print the total number of outliers for each feature
for column, count in outlier_counts.items():
    print(f'{column}: {count} outliers')

# Visualisation of outliers for each feature
plt.figure(figsize=(15, 10))
for i, column in enumerate(data.columns):
    if 'outlier' not in column:
        plt.subplot((len(data.columns) + 1) // 2, 2, i + 1)
        sns.boxplot(x=data[column])
        plt.title(f'Boxplot of {column}')
plt.tight_layout()
plt.show()

In [ ]:
# 5.3 Identify the number of features that must simultaneously be in outlier
# condition, in order for a sample to be classified as an outlier, such that the
# total percentage of samples identified as outliers falls within the 1-5% range

np.random.seed(0)  # For reproducibility

# Function to identify observations with outliers in at least N features
def identify_outliers_by_feature_count(data, n_features_required):
    # Sum the outlier flags for each observation
    outlier_columns = [col for col in data.columns if col.endswith('_is_outlier')]
    data['total_outliers'] = data[outlier_columns].sum(axis=1)

    # Filter observations with at least n_features_required outlier features
    observations_with_n_outliers = data[data['total_outliers'] >= n_features_required]

    # Calculate the percentage of such observations
    total_samples = len(data)
    outlier_percentage = (len(observations_with_n_outliers) / total_samples) * 100

    return observations_with_n_outliers, outlier_percentage

# Identify outliers with number of features flagged
for n in range(1, len(data.columns) - len(data.columns) // 2 + 1):
    obs_with_n_outliers, percentage = identify_outliers_by_feature_count(data, n)
    print(f'Number of features required: {n}, Outlier Percentage: {percentage:.2f}%, Number of Outliers: {len(obs_with_n_outliers)}')

# Loop through to find the number of features that results in 1-5% of samples being outliers
for num_features_required in range(1, len(data.columns) - len(data.columns) // 2 + 1):
    obs_with_n_outliers, outlier_percentage = identify_outliers_by_feature_count(data, num_features_required)

    if 1 <= outlier_percentage <= 5:
        print(f"Number of features that must simultaneously be in outlier condition for a sample to be classified as an outlier: {num_features_required}")
        print(f"Percentage of samples classified as outliers: {outlier_percentage:.2f}%")
        print(f"Number of outlier samples: {len(obs_with_n_outliers)}")
        print()
        break
else:
    print("No condition found where 1-5% of samples are outliers.")
    print()

<font color="blue">**A new binary column was created for each feature, indicating whether values are outliers based on IQR calculations.**</font>

In [ ]:
# print(data.head(10))

In [ ]:
data.columns

In [ ]:
# 5.2 Recalculating by using the flags only
data_recal = data.copy()
data_recal = data_recal[['Engine rpm_is_outlier', 'Lub oil pressure_is_outlier', 'Fuel pressure_is_outlier', 'Coolant pressure_is_outlier', 'lub oil temp_is_outlier', 'Coolant temp_is_outlier' ]]
data_recal.columns

In [ ]:
# Sum the flagged columns for each observation
outliers_recalculated = (data_recal == 1).sum(axis=1) # here includes boolean value "True"
print()
outliers_recalculated.describe()
print()
outliers_recalculated.info()

In [ ]:
# Create a mask for observations with at least two flagged columns
mask = outliers_recalculated >= 2

# Filter the DataFrame to get observations that meet the criteria
observations_with_at_least_two_flags = data[mask]

total_true_observations = mask.sum()

# Print the result
print(f'Total number of observations where mask is True: {total_true_observations}')

### <font color="blue">**PART III: Anomaly detection: Machine learning (ML) models**</font>

<font color="blue">**Features in the dataset are scaled to improve model performance**</font>

In [ ]:
# 6.1 Features in the dataset are scaled to improve model performance
data_copy = data.copy()

columns_to_drop = ['outliers_count_95th', 'total_outliers', 'Engine rpm_is_outlier', 'Lub oil pressure_is_outlier', 'Fuel pressure_is_outlier', 'Coolant pressure_is_outlier', 'lub oil temp_is_outlier', 'Coolant temp_is_outlier' ]

data_copy = data_copy.drop(columns=columns_to_drop)
print(data_copy.columns)

# Initialise the StandardScaler
standard_scaler = StandardScaler()

# Scale the data
data_scaled = standard_scaler.fit_transform(data_copy)

# Convert the scaled array back to a DataFrame
data_scaled_df = pd.DataFrame(data_scaled, columns=data_copy.columns)

# Number of columns for subplots
n_cols = 3
n_features = len(data_scaled_df.columns)
n_rows = (n_features + n_cols - 1) // n_cols  # Calculate number of rows required

# Create subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten()

# Plot histograms for each feature
for i, column in enumerate(data_scaled_df.columns):

  # Plot histogram
    sns.histplot(data_scaled_df[column], kde=True, color='red', edgecolor='black', ax=axes[i])
    axes[i].set_title(f'Histogram of {column}', fontsize=12)
    axes[i].set_xlabel(column, fontsize=10)
    axes[i].set_ylabel('Frequency', fontsize=10)

# Remove any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

In [ ]:
print(data_scaled_df.head())
data_scaled_df.info()

In [ ]:
# 6.1 One-class SVM was used for outlier detection to explore various approaches
model = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05)
model.fit(data_scaled)

In [ ]:
# Predict the class (normal or outlier)
y_pred = model.predict(data_scaled)
print(y_pred)

In [ ]:
# Create DataFrame
data_scaled_df['Outlier'] = y_pred
print(data_scaled_df.head())

In [ ]:
# Display outliers in the DataFrame.
anomalies = data_scaled_df[data_scaled_df.Outlier == -1]

# View output.
print(anomalies.shape)
print()
print(f"Number of anomalies inthis calculation indicates it is around 5% of the total sample: {anomalies.shape[0]}")
print()
print()
print(anomalies)

In [ ]:
# Use all features for fitting the model
X = data_scaled_df[['Engine rpm', 'Lub oil pressure', 'Fuel pressure', 'Coolant pressure', 'lub oil temp', 'Coolant temp']]

# Fit the One-Class SVM model
model = OneClassSVM(kernel='rbf', gamma=0.5, nu=0.05)
model.fit(X)
y_pred = model.predict(X)
data_scaled_df['Outlier'] = y_pred

# Print the value counts of predictions
print(data_scaled_df['Outlier'].value_counts())

# Function to plot the model for visualisation
def plot_svm_model(df, model, feature_pairs):
    n_pairs = len(feature_pairs)
    fig, axes = plt.subplots(nrows=n_pairs, figsize=(10, 6 * n_pairs), constrained_layout=True)

    if n_pairs == 1:
        axes = [axes]

    for ax, (feature1, feature2) in zip(axes, feature_pairs):
        x_min, x_max = df[feature1].min() - 1, df[feature1].max() + 1
        y_min, y_max = df[feature2].min() - 1, df[feature2].max() + 1

        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 500), np.linspace(y_min, y_max, 500))

        # Create meshgrid_df with all features, then select the two for plotting
        meshgrid_df = pd.DataFrame(index=range(len(xx.ravel())))
        for col in X.columns:
            if col == feature1:
                meshgrid_df[col] = xx.ravel()
            elif col == feature2:
                meshgrid_df[col] = yy.ravel()
            else:
                meshgrid_df[col] = 0  # Impute missing columns with 0 for prediction

        # Ensure meshgrid_df has the same column order as X
        meshgrid_df = meshgrid_df[X.columns]

        Z = model.decision_function(meshgrid_df)
        Z = Z.reshape(xx.shape)

        cf = ax.contourf(xx, yy, Z, levels=np.linspace(Z.min(), 0, 7), cmap=plt.cm.PuBu)
        ax.contour(xx, yy, Z, levels=[0], linewidths=2, colors='black')

        scatter = sns.scatterplot(data=df, x=feature1, y=feature2, hue='Outlier', palette={1: 'blue', -1: 'red'}, legend='full', ax=ax)
        scatter.legend(loc='upper right')

        ax.set_title(f'One-Class SVM with {feature1} vs {feature2}', fontsize=10)
        ax.set_xlabel(feature1, fontsize=8)
        ax.set_ylabel(feature2, fontsize=8)

    # Add a colorbar for the contour plot
    cbar = fig.colorbar(cf, ax=axes, orientation='vertical', fraction=0.02, pad=0.04)
    cbar.set_label('Decision Function Value', fontsize=8)

    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, ['Outlier', 'Inlier'], title='Outlier/Inlier', loc='upper right')
    plt.show()

# Generate all pairs of features
feature_pairs = list(combinations(X.columns, 2))

# Plot the One-Class SVM with Anomaly Detection for all feature pairs
plot_svm_model(data_scaled_df, model, feature_pairs)

<font color="blue">**Model settings for One-class SVM were adjusted to obtain the expected range of outliers**</font>

In [ ]:
# 3. Model settings for One-class SVM were adjusted to obtain the expected range of outliers

# Use all features for fitting the model
X = data_scaled_df[['Engine rpm', 'Lub oil pressure', 'Fuel pressure', 'Coolant pressure', 'lub oil temp', 'Coolant temp']]

# Adjust the One-Class SVM model parameters to get the expected range of outliers
model = OneClassSVM(kernel='rbf', gamma=0.1, nu=0.02)
model.fit(X)
y_pred = model.predict(X)
data_scaled_df['Outlier'] = y_pred

# Print the value counts of predictions
print(data_scaled_df['Outlier'].value_counts())

# Function to plot the model for visualization
def plot_svm_model(df, model, feature_pairs):
    n_pairs = len(feature_pairs)
    fig, axes = plt.subplots(nrows=n_pairs, figsize=(12, 6 * n_pairs), constrained_layout=True)

    if n_pairs == 1:
        axes = [axes]

    for ax, (feature1, feature2) in zip(axes, feature_pairs):
        x_min, x_max = df[feature1].min() - 1, df[feature1].max() + 1
        y_min, y_max = df[feature2].min() - 1, df[feature2].max() + 1

        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 500), np.linspace(y_min, y_max, 500))

        # Create meshgrid_df with all features, then select the two for plotting
        meshgrid_df = pd.DataFrame(index=range(len(xx.ravel())))
        for col in X.columns:
            if col == feature1:
                meshgrid_df[col] = xx.ravel()
            elif col == feature2:
                meshgrid_df[col] = yy.ravel()
            else:
                meshgrid_df[col] = 0  # Impute missing columns with 0 for prediction

        # Ensure meshgrid_df has the same column order as X
        meshgrid_df = meshgrid_df[X.columns]

        Z = model.decision_function(meshgrid_df)
        Z = Z.reshape(xx.shape)

        cf = ax.contourf(xx, yy, Z, levels=np.linspace(Z.min(), 0, 7), cmap=plt.cm.PuBu)
        ax.contour(xx, yy, Z, levels=[0], linewidths=2, colors='black')

        scatter = sns.scatterplot(data=df, x=feature1, y=feature2, hue='Outlier', palette={1: 'blue', -1: 'red'}, legend='full', ax=ax)
        scatter.legend(loc='upper right')

        ax.set_title(f'One-Class SVM with {feature1} vs {feature2}', fontsize=10)
        ax.set_xlabel(feature1, fontsize=8)
        ax.set_ylabel(feature2, fontsize=8)

    # Add a colorbar for the contour plot
    cbar = fig.colorbar(cf, ax=axes, orientation='vertical', fraction=0.02, pad=0.04)
    cbar.set_label('Decision Function Value', fontsize=8)

    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, ['Outlier', 'Inlier'], title='Outlier/Inlier', loc='upper right')
    plt.show()

# Generate all pairs of features
feature_pairs = list(combinations(X.columns, 2))

# Plot the One-Class SVM with Anomaly Detection for all feature pairs
plot_svm_model(data_scaled_df, model, feature_pairs)

<font color="blue">**Visualise the output in 2D after performing PCA and ensure the outliers are in a different colour**</font>

In [ ]:
# 6.2 Applying PCA on scaled DataFrame
pca = PCA(n_components=2)

# Fit and transform the data using PCA
data_pca = pca.fit_transform(data_scaled_df.drop(columns=['Outlier']))

# Convert the PCA-transformed data back to a DataFrame
data_pca_df = pd.DataFrame(data_pca, columns=['Principal Component 1', 'Principal Component 2'])

# Add the Outlier predictions to the PCA DataFrame
data_pca_df['Outlier'] = data_scaled_df['Outlier']
print(data_pca_df.head())

In [ ]:
# Comparing the shape of 'data_scaled' and PCA data
print(f"Shape of 'data_scaled': {data_scaled.shape}")
print()
print(data_pca_df.head()) # Print the first 5 rows of the PCA data
print()

# Convert the PCA data back to a DataFrame with component names
pca_columns = [f'Principal Component {i+1}' for i in range(data_pca.shape[1])]
data_pca_df = pd.DataFrame(data_pca, columns=pca_columns)
print(f"Shape of PCA data: {data_pca.shape}")
print()
print(data_pca_df.head())

<font color="blue">After implementing Principal Component Analysis (PCA) on the scaled DataFrame, the number of dimensions was reduced from 6 to 2.</font>

<font color="blue">PC1 and PC2 still contains most of the information in the large set.</font>

In [ ]:
# Get the loadings (coefficients of the principal components)
loadings = pd.DataFrame(pca.components_.T, columns=pca_columns, index=data_copy.columns)

# Print the loadings to see the contribution of each feature to the principal components
print("\nLoadings (Feature Contributions to Principal Components):")
print()
print(loadings)
# outputs the DataFrame created below, showing how much each original feature
# contributes to the principal components %%

In [ ]:
print(data_scaled_df.head())

In [ ]:
# Set the y_subset for coloring points
y_subset = data_scaled_df['Outlier']

# Define a custom colormap with red and blue colors
colors = ListedColormap(['red', 'blue'])

plt.figure(figsize=(12, 9))
scatter = plt.scatter(data_pca_df['Principal Component 1'], data_pca_df['Principal Component 2'], c=y_subset, cmap=colors, alpha=0.5)

plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.title('PCA of scaled dataset in 2D', fontsize=12)

plt.legend(handles=[plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=10, label='Normal'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label='Outlier')],
    title='Class', loc='upper right')

# Adding a colorbar to show the class distribution
plt.colorbar(scatter, ticks=[-1, 1], label='Outlier')

# Show plot
plt.show()

<font color="blue">**Interpretation:**</font>

* Data points vary from red to blue, representing a scaled variable (-1 to 1)
* The data clusters suggests significant variance captured by two components
* Some points are distant, indicating potential anomalies or distinct groups
* Component 1 is more influential, capturing more data variance( -4 to 10+)

<font color="blue">**PCA of Dataset with One-Class SVM Anomaly Detection**</font>

In [ ]:
# 6.2 Initialise the One-Class SVM model with default parameters
model = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05)

# Fit the model on the scaled data
model.fit(data_scaled_df)

# Predict anomalies (1 for normal, -1 for outlier)
predictions = model.predict(data_scaled_df)

# Add the predictions to the DataFrame
data_scaled_df['Outlier'] = predictions

# Initialise PCA, reducing the data to 2 principal components
pca = PCA(n_components=2)

# Fit and transform the data using PCA
data_pca = pca.fit_transform(data_scaled_df.drop(columns=['Outlier']))

# Convert the PCA-transformed data back to a DataFrame
data_pca_df = pd.DataFrame(data_pca, columns=['Principal Component 1', 'Principal Component 2'])

# Add the Outlier predictions from OCSVM to the PCA DataFrame
data_pca_df['Outlier'] = data_scaled_df['Outlier']

# Define colors for the plot
colors = {1: 'blue', -1: 'red'}

# Create the scatter plot
plt.figure(figsize=(10, 8))
scatter = plt.scatter(data_pca_df['Principal Component 1'], data_pca_df['Principal Component 2'], c=data_pca_df['Outlier'].apply(lambda x: colors[x]), alpha=0.5)

# Adding labels and title
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA of Dataset with One-Class SVM Anomaly Detection')

# Adding a legend to show class distribution
plt.legend(handles=[plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=10, label='Normal'),
                    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label='Outlier')],
           title='Class', loc='upper right')
# Show plot
plt.show()

# 6.3 Define a range of nu values to test
nu_values = np.linspace(0.01, 0.05, 5)  # 1% to 5%

for nu in nu_values:
    # Initialise OCSVM with the current nu value
    model = OneClassSVM(kernel='rbf', gamma='scale', nu=nu)

    # Fit the model and predict anomalies
    model.fit(data_scaled_df)
    predictions = model.predict(data_scaled_df)
    outlier_count = np.sum(predictions == -1)
    outlier_percentage = (outlier_count / len(predictions)) * 100

    print(f'nu: {nu:.2f}, Outlier Count: {outlier_count}, Outlier Percentage: {outlier_percentage:.2f}%')

    # If within 1-5% range, visualise the results
    if 1 <= outlier_percentage <= 5:
        data_scaled_df['Outlier'] = predictions
        data_pca_df['Outlier'] = data_scaled_df['Outlier']

        plt.figure(figsize=(10, 8))
        scatter = plt.scatter(data_pca_df['Principal Component 1'], data_pca_df['Principal Component 2'], c=data_pca_df['Outlier'].apply(lambda x: colors[x]), alpha=0.5)

        plt.xlabel('Principal Component 1')
        plt.ylabel('Principal Component 2')
        plt.title(f'PCA of Dataset with One-Class SVM (nu: {nu:.2f})')

        plt.legend(handles=[plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=10, label='Normal'),
                            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label='Outlier')],
                   title='Class', loc='upper right')
        plt.show()

<font color="blue">**Isolation Forest alone**</font>

In [ ]:
# Use all features for fitting the model
X = data_scaled_df[['Engine rpm', 'Lub oil pressure', 'Fuel pressure', 'Coolant pressure', 'lub oil temp', 'Coolant temp']]
print(X.head())

# Fit the Isolation Forest model
model = IsolationForest(contamination='auto', random_state=42)
model.fit(X)
y_pred = model.predict(X)
data_scaled_df['Outlier'] = y_pred

# Print the value counts of predictions
print(data_scaled_df['Outlier'].value_counts())

In [ ]:
print(data_scaled_df.head())

In [ ]:
# Use all features for fitting the model
X = data_scaled_df[['Engine rpm', 'Lub oil pressure', 'Fuel pressure', 'Coolant pressure', 'lub oil temp', 'Coolant temp']]

# Fit the Isolation Forest model
model = IsolationForest(contamination=0.05, random_state=42)
model.fit(X)
y_pred = model.predict(X)
data_scaled_df['Outlier'] = y_pred

# Print the value counts of predictions
print(data_scaled_df['Outlier'].value_counts())

<font color="blue">**PCA with Isolation Forest**</font>

In [ ]:
# Initialise PCA, reducing the data to 2 principal components
pca = PCA(n_components=2)

# Fit and transform the data using PCA
data_pca = pca.fit_transform(data_scaled_df.drop(columns=['Outlier']))

# Convert the PCA-transformed data back to a DataFrame
data_pca_df = pd.DataFrame(data_pca, columns=['Principal Component 1', 'Principal Component 2'])

# Add the Outlier predictions to the PCA DataFrame
data_pca_df['Outlier'] = data_scaled_df['Outlier']

# Define colors for the plot
colors = {1: 'blue', -1: 'red'}

# Create the scatter plot
plt.figure(figsize=(10, 8))
scatter = plt.scatter(data_pca_df['Principal Component 1'], data_pca_df['Principal Component 2'], c=data_pca_df['Outlier'].apply(lambda x: colors[x]), alpha=0.5)

# labels and title
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA of Dataset with Isolation Forest Anomaly Detection')

# Adding a legend to show class distribution
plt.legend(handles=[plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=10, label='Normal'),
                    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label='Outlier')],
           title='Class', loc='upper right')

plt.show()

# Define a range of contamination values to test
contamination_values = np.linspace(0.01, 0.05, 5)  # 1% to 5%

for contamination in contamination_values:
    # Initialise Isolation Forest with the current contamination value
    isolation_forest = IsolationForest(contamination=contamination, random_state=42)

    # Fit the model and predict anomalies
    isolation_forest.fit(data_scaled_df)
    predictions = isolation_forest.predict(data_scaled_df)
    outlier_count = np.sum(predictions == -1)
    outlier_percentage = (outlier_count / len(predictions)) * 100

    print(f'Contamination: {contamination:.2f}, Outlier Count: {outlier_count}, Outlier Percentage: {outlier_percentage:.2f}%')

    # If within 1-5% range, visualise the results
    if 1 <= outlier_percentage <= 5:
        data_scaled_df['Outlier'] = predictions
        data_pca_df['Outlier'] = data_scaled_df['Outlier']

        plt.figure(figsize=(10, 8))
        scatter = plt.scatter(data_pca_df['Principal Component 1'], data_pca_df['Principal Component 2'], c=data_pca_df['Outlier'].apply(lambda x: colors[x]), alpha=0.5)

        plt.xlabel('Principal Component 1')
        plt.ylabel('Principal Component 2')
        plt.title(f'PCA of Dataset with Isolation Forest (Contamination: {contamination:.2f})')

        plt.legend(handles=[plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=10, label='Normal'),
                            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label='Outlier')],
                   title='Class', loc='upper right')
        plt.show()